In [1]:
import pickle
import geopandas as gpd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report

# Load the feature matrix
wards = gpd.read_file("../../Data/floods.gpkg")

# Create elevation range feature
wards['elevation_range_m'] = wards['elevation_max_m'] - wards['elevation_min_m']

# Drop geometry and metadata for modelling — keep only numeric features and target
feature_cols = [
    'pop2009',
    'rain_cumulative_mm',
    'rain_max_daily_mm',
    'rain_preflood_7d_mm',
    'elevation_mean_m',
    'elevation_min_m',
    'elevation_max_m',
    'elevation_range_m',
    'slope_mean_deg'
]

X = wards[feature_cols]
y = wards['flooded']

print(f"Feature matrix shape : {X.shape}")
print(f"Class distribution   :\n{y.value_counts(normalize=True)}")
print(f"\nMissing values:\n{X.isnull().sum()}")

Feature matrix shape : (1450, 9)
Class distribution   :
flooded
0    0.788276
1    0.211724
Name: proportion, dtype: float64

Missing values:
pop2009                0
rain_cumulative_mm     0
rain_max_daily_mm      0
rain_preflood_7d_mm    0
elevation_mean_m       0
elevation_min_m        0
elevation_max_m        0
elevation_range_m      0
slope_mean_deg         0
dtype: int64


In [2]:
# Perform train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=2026, stratify=y)
X_train.head()

,pop2009,rain_cumulative_mm,rain_max_daily_mm,rain_preflood_7d_mm,elevation_mean_m,elevation_min_m,elevation_max_m,elevation_range_m,slope_mean_deg
454,20756.0,229.351028,14.185050,27.096939,1255.182093,1164.0,1404.0,240.0,2.742085
223,25605.0,0.000000,0.000000,0.000000,1561.165994,1471.0,1630.0,159.0,4.014922
1347,26216.0,74.334179,15.210496,12.359124,270.326545,197.0,327.0,130.0,0.683518
841,27830.0,158.794473,24.094767,15.352821,1616.746939,1344.0,1972.0,628.0,4.300813
718,27731.0,137.708249,12.970549,27.609018,1903.645283,1661.0,2169.0,508.0,9.891995


In [3]:
# Scale features using StandardScaler
from sklearn.preprocessing import StandardScaler

# Instantiate the scaler and fit-transform the training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Transform the test data using the same scaler
X_test_scaled = scaler.transform(X_test)

In [4]:
# Build a simple XGBoost model
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

# Train the model
model.fit(X_train_scaled, y_train)

# Predict on the test set
y_pred = model.predict(X_test_scaled)

# Evaluate the model
print("Classification Report:")
print(classification_report(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.93      0.91       343
           1       0.69      0.60      0.64        92

    accuracy                           0.86       435
   macro avg       0.79      0.76      0.78       435
weighted avg       0.85      0.86      0.85       435



In [5]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

# Create a pipeline with SMOTE, scaling, and XGBoost
pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=2026)),
    ('scaler', StandardScaler()),
    ('classifier', XGBClassifier(random_state=42))
])

# Define parameters for GridSearchCV using pipeline prefixes
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.01, 0.1]
}

# Perform grid search with cross-validation
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1
)

# Fit the grid search to the raw training data
grid_search.fit(X_train, y_train)

# Best parameters and model
print("Best Parameters:", grid_search.best_params_)
best_model = grid_search.best_estimator_

# Predict on the test set with the best model
y_pred_best = best_model.predict(X_test)

# Evaluate the best model
print("Classification Report for Best Model:")
print(classification_report(y_test, y_pred_best))

Best Parameters: {'classifier__learning_rate': 0.01, 'classifier__max_depth': 7, 'classifier__n_estimators': 200}
Classification Report for Best Model:
              precision    recall  f1-score   support

           0       0.95      0.81      0.87       343
           1       0.54      0.83      0.65        92

    accuracy                           0.81       435
   macro avg       0.74      0.82      0.76       435
weighted avg       0.86      0.81      0.83       435



Saving the model:

In [6]:
with open('../best_xg_boost_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print("Best XGBoost model saved")

Best XGBoost model saved
